# Algorithms 34: Decision Trees**Learning Objectives:**1. Formulate rules-based classification as a Tree structure2. Compute Shannon Entropy to measure dataset impurity3. Implement the recursive ID3 / CART algorithm to automatically grow a Decision Tree**Prerequisites:** Algorithms 29**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapter 34

In [ ]:
# Google Colab Setupimport mathfrom collections import dequeimport heapqprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### Rules-Based ClassificationKNN stores all data points. A **Decision Tree** compresses the data into a flowchart of Yes/No questions. For example, to predict whether a customer will buy a product:- *Is Age > 30?*  - Yes: *Is Income > 50k?* -> Predict: **BUY**  - No: Predict: **DO NOT BUY**### How does the algorithm choose the questions?It uses **Information Theory**. We want to ask the question that perfectly splits the data into pure groups (all BUYs on one side, all DO NOT BUYs on the other).We measure impurity using **Shannon Entropy ($H$)**:$$H = - \sum p_i \log_2(p_i)$$where $p_i$ is the probability of class $i$ in the current subset of data.- If a group is 100% BUY, $H = 0$ (Perfectly pure, zero chaos/information).- If a group is 50% BUY / 50% DO NOT BUY, $H = 1$ (Maximum chaos/information).### Information GainThe algorithm tests every possible split, and calculates the weighted average Entropy of the two resulting branches. **Information Gain = Current Entropy - Weighted Branch Entropy**The algorithm greedily picks the split with the maximum Information Gain, and then recursively applies the process to the branches!

### Comprehension Check ✅Calculate the Entropy of a dataset with 4 cats, 4 dogs. Now calculate the Entropy of a dataset with 8 cats, 0 dogs.<details><summary>💡 Answers</summary>1. 4 cats, 4 dogs: $p_{cat} = 0.5, p_{dog} = 0.5$. $H = -(0.5 \log_2(0.5) + 0.5 \log_2(0.5)) = -(-0.5 - 0.5) = 1.0$2. 8 cats: $p_{cat} = 1.0, p_{dog} = 0.0$. (By definition $0 \log(0) = 0$).$H = -(1.0 \log_2(1.0) + 0) = 0.0$</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: Shannon Entropy FunctionWrite a function `entropy(labels)` that takes a list of class labels (e.g. `['cat', 'cat', 'dog']`) and computes $H$.

In [ ]:
import mathfrom collections import Counterdef entropy(labels):    if not labels: return 0.0    counts = Counter(labels)    h = 0.0    # For each count in counts.values():    # p = count / len(labels)    # h -= p * math.log2(p)    # YOUR CODE HERE    return h# assert entropy(['cat', 'dog', 'cat', 'dog']) == 1.0# assert entropy(['cat', 'cat']) == 0.0

---## Phase 1 — Algorithm Construction### Step 1: Splitting the DataWrite a function `split_data(data, feature_idx, threshold)` that splits a dataset based on a condition.`data` is a list of `(features_tuple, label)`. Return `(left_data, right_data)` where `left` is $\le$ threshold, `right` is $>$ threshold.

In [ ]:
def split_data(data, feature_idx, threshold):    left, right = [], []    # YOUR CODE HERE    return left, right

### Step 2: Finding the Best SplitWrite a function `find_best_split(data)` that loops through every feature, and every possible threshold (every value seen in the dataset for that feature), and calculates Information Gain. Return the `(best_feature_idx, best_threshold)`. *(If the data is completely pure, return None)*

In [ ]:
def find_best_split(data):    best_gain = 0.0    best_split = None    current_entropy = entropy([label for _, label in data])        if current_entropy == 0: return None        num_features = len(data[0][0])        for f_idx in range(num_features):        # Get all unique values for this feature to try as thresholds        thresholds = set(row[0][f_idx] for row in data)                for t in thresholds:            left, right = split_data(data, f_idx, t)            if not left or not right: continue                        # Calculate weighted average entropy of children            p_left = len(left) / len(data)            p_right = len(right) / len(data)            child_entropy = p_left * entropy([l for _, l in left]) + p_right * entropy([l for _, l in right])                        gain = current_entropy - child_entropy            if gain > best_gain:                best_gain = gain                best_split = (f_idx, t)                    return best_split

### Step 3: Building the Tree RecursivelyLet's define a `Node` class. Then `build_tree(data)` will recursively call `find_best_split`.

In [ ]:
class Node:    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, prediction=None):        self.feature_idx = feature_idx        self.threshold = threshold        self.left = left        self.right = right        self.prediction = prediction # Only set for leaf nodesdef build_tree(data, max_depth=5, current_depth=0):    # 1. Base cases: pure data, or max depth reached    labels = [l for _, l in data]    if len(set(labels)) == 1 or current_depth == max_depth:        return Node(prediction=Counter(labels).most_common(1)[0][0])            # 2. Find best split    split = find_best_split(data)    if not split:        return Node(prediction=Counter(labels).most_common(1)[0][0])            f_idx, t = split    left_data, right_data = split_data(data, f_idx, t)        # 3. Recursively build branches    left_node = build_tree(left_data, max_depth, current_depth + 1)    right_node = build_tree(right_data, max_depth, current_depth + 1)        return Node(feature_idx=f_idx, threshold=t, left=left_node, right=right_node)

### Step 4: InferenceWrite `predict_tree(node, point)` which drops a single point down the tree until it hits a leaf.

In [ ]:
def predict_tree(node, point):    # If it's a leaf node, return prediction    # YOUR CODE HERE    pass    # Otherwise, check point[node.feature_idx] against node.threshold    # Recursively call predict_tree on left or right child    # YOUR CODE HERE

---## Phase 2 — Experimental Verification### The XOR ProblemLet's test the tree on a dataset that linear models (like standard pseudoinverse) fail catastrophically on: the XOR logic gate!

In [ ]:
xor_data = [    ((0, 0), 0), ((0, 1), 1),    ((1, 0), 1), ((1, 1), 0)]# root = build_tree(xor_data)# print("Predict (0,0):", predict_tree(root, (0,0))) # Should be 0# print("Predict (0,1):", predict_tree(root, (0,1))) # Should be 1# print("Predict (1,0):", predict_tree(root, (1,0))) # Should be 1# print("Predict (1,1):", predict_tree(root, (1,1))) # Should be 0

---📚 **Next:** Algorithms 35-36 (Neural Networks)